In [1]:
# On exécute le précédent notebook car on est dépendant des résultats
import sys
import os
from contextlib import redirect_stdout, redirect_stderr

# Désactiver l'affichage AVANT d'importer matplotlib
%matplotlib agg

with open(os.devnull, 'w') as fnull:
    with redirect_stdout(fnull), redirect_stderr(fnull):
        %run 1_analyse-exploratoire.ipynb

# FORCER la réactivation de l'affichage
#%matplotlib inline

# Réinitialiser matplotlib pour forcer le nouveau backend
import matplotlib
matplotlib.pyplot.close('all')  # Fermer toutes les figures existantes

In [2]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import numpy as np

scaler = MinMaxScaler()


# Feature Engineering

## Définition de la cible

Avant de créer les features, nous allons définir les variables cibles.

* Consommation totale en énergie (tout type confondu) : `SiteEnergyUse(kBtu)`
* Total des émissions de gaz à effet de serre : `TotalGHGEmissions`

In [3]:
def show_target_distribution():
    # Conversion kBtu -> MBtu
    df['SiteEnergyUse(MBtu)'] = df['SiteEnergyUse(kBtu)'] / 1000
    
    plt.figure(figsize=(12, 5))
    
    # Émissions de GES
    plt.subplot(1, 2, 1)
    sns.histplot(df['TotalGHGEmissions'].dropna(), bins=40, color='skyblue', edgecolor='black')
    plt.xlabel('Émissions totales de GES (kgCO2e)')
    plt.ylabel('Effectif (bâtiments)')
    plt.title('Distribution des émissions de GES')
    
    # Consommation d’énergie en MBtu
    plt.subplot(1, 2, 2)
    sns.histplot(df['SiteEnergyUse(MBtu)'].dropna(), bins=40, color='salmon', edgecolor='black')
    plt.xlabel('Consommation totale d’énergie (MBtu)')
    plt.ylabel('Effectif (bâtiments)')
    plt.title('Distribution de la consommation d’énergie')
    # Formatage des grands nombres avec espace tous les 3 chiffres
    plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: format(int(x), ',').replace(',', ' ')))
    
    plt.tight_layout()
    plt.show()

show_target_distribution()

On constate une **grande asymétrie** avec des outliers à droite. On va retirer 1% percentile à droite et garder les 99% percentile à gauche pour retirer les plus grands outliers tout en gardant un maximum de données. Après une petite analyse ces données ne me semblent pas complètement abbérantes car il s'agit de très grandes surfaces.

In [4]:
# Calculer les percentiles 99e pour les deux variables cibles
ghg_99 = df['TotalGHGEmissions'].quantile(0.99)
energy_99 = df['SiteEnergyUse(MBtu)'].quantile(0.99)

# Supprimer les 1% à droite pour chaque variable cible
df_clean = df[(df['TotalGHGEmissions'] <= ghg_99) & (df['SiteEnergyUse(MBtu)'] <= energy_99)]

print(f"Taille du dataset après suppression des outliers : {len(df_clean)} lignes")
print(f"Nombre de lignes supprimées : {len(df) - len(df_clean)}")
print(f"Affichage de la nouvelle distribution (zoom))")

df = df_clean

show_target_distribution()

Taille du dataset après suppression des outliers : 1507 lignes
Nombre de lignes supprimées : 19
Affichage de la nouvelle distribution (zoom))


## Besoins en données
On a besoin à minima des données suivantes : 

* Surface totale
* Localisation
* Usage du bâtiment
* Nombre de bâtiments et / ou d'étages (surface exposée à l'air)
* Année de construction
* Type de source d'énergie
  
## Features

### Surface totale
  * `PropertyGFATotal` : on utilise MinMaxScaler pour diminuer la grande amplitude des données

In [5]:
df['PropertyGFATotalScaled'] = scaler.fit_transform(df[['PropertyGFATotal']])

### Localisation
  * `Neighborhood` : information pertinente en terme de localisation, car elle peut englober un facteur d'architecture d'urbanisme qui fera potentiellement varier les résultats (exemple : dépendance de tout un quartier à une infrastructure électrique comme un ou plusieurs transformateurs). On utilise un **OneHotEncoder** car il s'agit d'une catégorie non ordinale



In [6]:
# Appliquer le One-Hot Encoding sur Neighborhood, d'après l'analyse précédente on va limiter à 12 le nombre de catégories
encoder = OneHotEncoder(sparse_output=False, max_categories=12)
neighborhood_encoded = encoder.fit_transform(df[['Neighborhood']])

# Ajouter les colonnes encodées directement au DataFrame
df = pd.concat([df.drop('Neighborhood', axis=1),
                pd.DataFrame(neighborhood_encoded,
                           columns=encoder.get_feature_names_out(['Neighborhood']),
                           index=df.index)], axis=1)

# Vérification
neighborhood_columns = [col for col in df.columns if col.startswith('Neighborhood_')]
print(f"Nombre de nouvelles colonnes Neighborhood_ créées : {len(neighborhood_columns)}")

Nombre de nouvelles colonnes Neighborhood_ créées : 12



  * `Lat` / `Lng` : ces données de localisation sont très fines mais le modèle ne comprendra naturellement pas l'aspect géographique. On va grouper les données (clusters) afin de capter des effets locaux / zones caractéristiques.

In [7]:
# Appliquer le K-means sur Lat & Lng
n_clusters = 8 # à ajuster selon la taille de la ville
coords = df[['Latitude', 'Longitude']]
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
df['GeoCluster'] = kmeans.fit_predict(coords)

# Affichage de la courbe GeoCluster 
# Visualiser la répartition géographique des clusters
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x='Longitude', y='Latitude', hue='GeoCluster', palette='tab10')
plt.title('Répartition géographique des clusters')
plt.show()

# Visualiser la distribution des variables cible par cluster
plt.figure(figsize=(10,6))
sns.boxplot(data=df, x='GeoCluster', y='TotalGHGEmissions')
plt.title('Distribution de TotalGHGEmissions par GeoCluster')
plt.show()

plt.figure(figsize=(10,6))
sns.boxplot(data=df, x='GeoCluster', y='SiteEnergyUse(kBtu)')
plt.title('Distribution de SiteEnergyUse(kBtu) par GeoCluster')
plt.show()



**Remarque : On constate qu'il n'y a pas de relation ordinale avec les clusters ce qui est tout à fait cohérent. On va utiliser un OneHotEncoder pour séparer les GeoClusters.**

In [8]:
encoder = OneHotEncoder(drop='first', sparse_output=False)  # drop='first' évite la multicolinéarité
geo_encoded = encoder.fit_transform(df[['GeoCluster']])
feature_names = encoder.get_feature_names_out(['GeoCluster'])
print(f"\nNoms des features: {feature_names}")



Noms des features: ['GeoCluster_1' 'GeoCluster_2' 'GeoCluster_3' 'GeoCluster_4'
 'GeoCluster_5' 'GeoCluster_6' 'GeoCluster_7']


### Usage du bâtiment

* `PrimaryPropertyType` est l'usage principal du bâtiment, cette variable est critique pour mesurer la consommation. On ne peut pas l'utiliser telle quelle, on va appliquer un OneHot Encoding.

In [9]:
# Appliquer le one-hot encoding sur PrimaryPropertyType
encoder = OneHotEncoder(sparse_output=False, drop=None)
encoded = encoder.fit_transform(df[['PrimaryPropertyType']])
df = pd.concat([df.drop(columns=['PrimaryPropertyType']),
                pd.DataFrame(encoded, columns=[f"Type_{cat}" for cat in encoder.categories_[0]], index=df.index)], axis=1)

# Vérifier le résultat
print(f"Nombre de colonnes d'usage primaire créées : {len([col for col in df.columns if col.startswith('Type_')])}")

Nombre de colonnes d'usage primaire créées : 19


* On crée une feature pour savoir si un bâtiment a **plusieurs** usages.

In [10]:
df['MultipleUsages'] = df['ListOfAllPropertyUseTypes'].str.contains(',', na=False)

### Nombre de bâtiments et / ou d'étages (surface exposée à l'air)

* Nombre de bâtiments `NumberofBuildings` : on constate qu'il y a quelques données outlier, notamment le `University of Washington - Seattle Campus` qui a une donnée extrême de **111** bâtiments. On fait le choix de garder cette donnée car elle est vérifiée, toutefois on pourra agir dessus si besoin selon les performances du modèle

In [11]:
# décommenter pour décrire et voir les outliers
# print(df[['NumberofFloors', 'NumberofBuildings']].describe())
# display((df['NumberofBuildings'] > 12).sum())

* Nombre d'étages `NumberofFloors` : on constate un outlier qui est une erreur : `Seattle Chinese Baptist Church` est notée comme ayant 99 étages, en réalité l'église semble n'avoir que 2 étages (vérifié sur le net). On corrige donc cette valeur abbérante.

In [12]:
# décommenter pour voir les bâtiments qui ont plus de 40 étages
# display(df[df['NumberofFloors'] > 40])

# correction du nombre d'étages pour l'église Chinese Baptiste
df.loc[df['OSEBuildingID'] == 21611, 'NumberofFloors'] = 2

### Année de construction

L'année de construction nous permet de déduire des consommations en énergie / C02 à partir de l'âge d'un bâtiment.
On va se servir de cette donnée pour créer une nouvelle feature `BuildingAge` **calculée à partir de la date de collecte des données**

**Attention : Bien que l'âge peut être très pertinent pour la prédiction, il va également entrainer du drift du modèle. Il faudra donc régulièrement réentrainer notre modèle si l'on utilise cette feature** 


In [13]:
print("=== ANALYSE DE LA VARIABLE YearBuilt ===")
print(f"Nombre total de bâtiments: {len(df)}")
print(f"Valeurs manquantes: {df['YearBuilt'].isna().sum()}")
print(f"Année minimum: {df['YearBuilt'].min()}")
print(f"Année maximum: {df['YearBuilt'].max()}")
print(f"Médiane: {df['YearBuilt'].median()}")

# Compter le nombre de bâtiments par année
year_counts = df['YearBuilt'].value_counts().sort_index()

# Créer l'histogramme
plt.figure(figsize=(15, 8))
bars = plt.bar(year_counts.index, year_counts.values, color='lightcoral', alpha=0.7, edgecolor='black', width=0.8)
plt.title('Répartition des bâtiments par année de construction', fontsize=16, fontweight='bold')
plt.ylabel('Nombre de bâtiments', fontsize=12)
plt.xlabel('Année de construction', fontsize=12)
plt.grid(axis='y', alpha=0.3)

# Rotation des labels pour une meilleure lisibilité
plt.xticks(rotation=45)

# Ajouter une ligne de tendance (moyenne mobile)
window = 5
if len(year_counts) >= window:
    moving_avg = year_counts.rolling(window=window, center=True).mean()
    plt.plot(moving_avg.index, moving_avg.values, color='red', linewidth=2, label=f'Moyenne mobile ({window} ans)')
    plt.legend()

plt.tight_layout()
plt.show()

# Calculer l'âge du bâtiment
df['BuildingAge'] = df['DataYear'] - df['YearBuilt']

# Histogramme de l'âge
plt.figure(figsize=(12, 7))
plt.hist(df['BuildingAge'], bins=30, color='mediumseagreen', edgecolor='black', alpha=0.7)
plt.title("Répartition de l'âge des bâtiments (en 2024)", fontsize=16, fontweight='bold')
plt.xlabel("Âge du bâtiment (années)", fontsize=12)
plt.ylabel("Nombre de bâtiments", fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

=== ANALYSE DE LA VARIABLE YearBuilt ===
Nombre total de bâtiments: 1507
Valeurs manquantes: 0
Année minimum: 1900
Année maximum: 2015
Médiane: 1965.0


### Types de sources d'énergie

On crée 2 features pour savoir si le bâtiment : 

* utilise une source électrique
* utilise une source de gaz

In [14]:
df['UsesElectricity'] = (df['Electricity(kBtu)'] > 0).astype(int)
df['UsesNaturalGas'] = (df['NaturalGas(kBtu)'] > 0).astype(int)

### Corrélation entre les features sélectionnées et variables cibles

In [15]:
# Liste des features numériques retenues (à adapter selon tes besoins)

neighborhood_cols = [col for col in df.columns if col.startswith('Neighborhood_')]
type_cols = [col for col in df.columns if col.startswith('Type_')]
geo_clusters_cols = [col for col in df.columns if col.startswith('GeoCluster_')]


# Variables cibles
target = [
    'TotalGHGEmissions',
    'SiteEnergyUse(kBtu)'
]

# Variables prédictives
features_selected = [
    'PropertyGFATotalScaled',
    'BuildingAge',
    'UsesElectricity',
    'UsesNaturalGas',
    'NumberofBuildings', 
    'NumberofFloors'
]

# Sélectionner uniquement les features retenues
df_selected = df[features_selected + neighborhood_cols + type_cols + geo_clusters_cols]
nb_total_features = len(features_selected) + len(neighborhood_cols) + len(type_cols) + len(geo_clusters_cols)
df_matrix_preview = df[features_selected]

# Calculer la matrice de corrélation
corr_matrix = df_matrix_preview.corr(method='pearson')

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', center=0)
plt.title("Matrice de corrélation de Pearson - réduite aux variables numériques")
plt.tight_layout()
plt.show()

print(f"Nombre total de features sélectionnées : {nb_total_features}")

Nombre total de features sélectionnées : 37


In [16]:
features = ['SiteEnergyUse(kBtu)', 'TotalGHGEmissions', 'PropertyGFATotal', 'BuildingAge']
sns.pairplot(df[features], kind="reg")
plt.show()